# 🏦 Project: DinDin do Bem - AI for Financial Inclusion
### *Behavioral Credit Scoring using Neural Networks and Alternative Data*

Author: Danielle Sismon

Discipline: Machine Learning

Course: Multiplatform Software Development - FATEC (São Paulo State Technology College)

## 🎯 Project Objective

This project simulates the development of the credit decision engine for **DinDin do Bem**, a fictional Fintech focused on the **financial inclusion of women** (especially widows, divorced women, and self-employed workers) who are made invisible by the traditional banking system.

The technical objective is to build a **Machine Learning model (MLP Neural Network)** capable of predicting the risk of default (*Credit Scoring*) using **Alternative Data** (behavior, stability, and social history) instead of relying solely on traditional financial history.

Using a robust database and Deep Learning techniques, we seek to build a decision system that analyzes behavioral variables (such as phone contact stability and domestic responsibility) to predict default risk. The focus is to balance the financial security of the institution (high Recall in detecting bad payers) with the social purpose of inclusion, proving that it is possible to grant credit fairly and technically.

## 💜 The Business Problem: "Thin File" and Vulnerability

In the traditional financial scenario, granting credit depends strictly on proven banking history (bureau score, pay stubs). This model excludes a significant portion of the population: vulnerable women (widows, divorced) or informal workers. Often, these women are "invisible" to big banks (a phenomenon known as the Thin File problem), which perpetuates cycles of financial dependency and, in severe cases, makes it difficult to break cycles of domestic violence. Data technology emerges as a tool for social justice, allowing the analysis of behavior (alternative data) instead of just formal income.

Many vulnerable women do not have a pay stub, formal employment, or a credit bureau history. This creates a vicious cycle:

1. The bank denies credit because she has no history.
2. Without credit, she builds no history and remains dependent (often in situations of financial abuse).

**DinDin do Bem** proposes breaking this cycle by analyzing **Behavior** (contact stability, domestic responsibility) as a proxy for reliability.

## 🛠️ Methodology and Technical Pipeline
The project follows a rigorous Data Science workflow:

1. **Data Collection:** Using the *Home Credit Default Risk* database (Kaggle), adapted for the Brazilian context.
   Link: https://www.kaggle.com/competitions/home-credit-default-risk/data

2. **Feature Engineering:** Selection of **Alternative Data** variables (E.g.: `DAYS_SINCE_PHONE_CHANGE` as a stability indicator).

3. **Data Treatment:**
   * Domain translation for better interpretability.
   * Missing value imputation (Median) to handle income outliers.
   * Categorical variable encoding (*One-Hot Encoding*).

4. **Class Balancing (SMOTE):** Application of the *Synthetic Minority Over-sampling Technique* to correct the natural imbalance between good and bad payers, avoiding model bias.

5. **Modeling (Deep Learning):** Training a **Neural Network (Multilayer Perceptron)** with a funnel architecture (100 -> 50 neurons) and ReLU activation.

6. **Evaluation:** Focus on business metrics (**Recall**) and visualization using Confusion Matrix and ROC Curve.

## 📚 Technologies Used
* **Language:** Python
* **Data Manipulation:** Pandas, NumPy
* **Machine Learning:** Scikit-Learn (MLPClassifier, Preprocessing)
* **Balancing:** Imbalanced-Learn (SMOTE)
* **Visualization:** Seaborn, Matplotlib

---
*This project was developed as a requirement for the Machine Learning discipline (Fatec), under the guidance of Prof. Meg Lima Andrade, and aims to demonstrate the ethical application of AI for social impact.*

In [ ]:
# 1.1 - Importing the necessary tools for this project:
import pandas as pd # To read and manipulate the table
import numpy as np # For mathematical calculations
import matplotlib.pyplot as plt # For charts
import seaborn as sns # For better-looking charts

# Machine Learning Tools:
from sklearn.model_selection import train_test_split # To separate study and test data
from sklearn.preprocessing import StandardScaler, OneHotEncoder # To translate data
from sklearn.impute import SimpleImputer # To fill holes (missing data)
from sklearn.compose import ColumnTransformer # To organize the treatment
from sklearn.pipeline import Pipeline # To create a production line
from sklearn.neural_network import MLPClassifier # THE CHOSEN NEURAL NETWORK!
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc # To grade the model
from imblearn.over_sampling import SMOTE # To create "clones" (data balancing)

# 1.2 - Loading the data:
print("Loading the database (please wait a moment)...") 

df = pd.read_csv('application_train.csv') 

print(f"\nDatabase loaded! We have {df.shape[0]} rows / people and {df.shape[1]} columns.") 

# 1.3 - Printing the names of all columns in the database:
print("\nLIST OF ALL AVAILABLE COLUMNS:\n") 
print(df.columns.tolist()) 

# 1.4 - Translating columns and values to English:
print("\nTranslating columns and values to English (please wait a moment)...\n") 

# i. Column Translation Dictionary:
column_map = {
    'CODE_GENDER': 'GENDER', 
    'TARGET': 'TARGET', # 0 = Paid | 1 = Default
    'NAME_HOUSING_TYPE': 'HOUSING_TYPE', # Rented, Owned, etc.
    'NAME_FAMILY_STATUS': 'MARITAL_STATUS', 
    'NAME_EDUCATION_TYPE': 'EDUCATION_LEVEL', 
    'DAYS_LAST_PHONE_CHANGE': 'DAYS_SINCE_PHONE_CHANGE', 
    'DAYS_ID_PUBLISH': 'DAYS_ID_PUBLISHED', 
    'DAYS_EMPLOYED': 'DAYS_EMPLOYED', 
    'AMT_INCOME_TOTAL': 'TOTAL_INCOME', 
    'AMT_CREDIT': 'LOAN_AMOUNT_REQUESTED', 
    'AMT_ANNUITY': 'INSTALLMENT_AMOUNT', 
    'EXT_SOURCE_1': 'SOCIAL_MICROCREDIT_SCORE', # Focuses on social reputation
    'EXT_SOURCE_2': 'BASIC_BILLS_REGISTRY_SCORE', # Focuses on basic bills responsibility
    'EXT_SOURCE_3': 'RETAIL_BEHAVIOR_SCORE', # Focuses on retail purchase history
    'NAME_CONTRACT_TYPE': 'CONTRACT_TYPE' 
} 

# Applying column renaming:
df.rename(columns=column_map, inplace=True) 

# ii. Values Translation Dictionary (cell content):
value_map = {
    # Gender
    'F': 'Female', 
    'M': 'Male', 
    'XNA': 'Undefined', 

    # Marital Status:
    'Married': 'Married', 
    'Single / not married': 'Single', 
    'Civil marriage': 'Civil Union', 
    'Separated': 'Separated / Divorced', 
    'Widow': 'Widow', 
    'Unknown': 'Unknown', 

    # Housing Type:
    'House / apartment': 'Owned House / Apt', 
    'Rented apartment': 'Rented', 
    'With parents': 'With Parents', 
    'Municipal apartment': 'Municipal Housing', 
    'Office apartment': 'Office Housing', 
    'Co-op apartment': 'Co-op Housing', 

    # Education:
    'Secondary / secondary special': 'High School', 
    'Higher education': 'Higher Education', 
    'Incomplete higher': 'Incomplete Higher Ed', 
    'Lower secondary': 'Middle School', 
    'Academic degree': 'Master / PhD' 
}

# 1.5 - Applying the translation across the entire DataFrame:
print("Applying complete translation...\n") 
df.replace(value_map, inplace=True) 

print("Database 100% translated! Checking a sample: \n ") 
print(df[['GENDER', 'MARITAL_STATUS', 'EDUCATION_LEVEL', 'HOUSING_TYPE']].head(3)) 

In [ ]:
# 2 - TECHNICAL EXPLORATORY ANALYSIS:
# Missing data, standard deviation, and types.

print("DATA RADIOGRAPHY:\n") 

# i. Checking missing data (Blank items)
analysis_columns = [
    'TOTAL_INCOME', 
    'LOAN_AMOUNT_REQUESTED', 
    'SOCIAL_MICROCREDIT_SCORE', 
    'DAYS_EMPLOYED' 
]

print(f"Amount of missing items (NaN) in key columns:") 
# The command below counts how many holes exist in each column
print(df[analysis_columns].isnull().sum()) 

# ii. Descriptive Statistics (Standard Deviation, Mean, Outliers)
# The .describe() command shows the mean and std
print("\nDescriptive Statistics (Mean, Std Dev, Min, Max):\n") 
print(df[['TOTAL_INCOME', 'LOAN_AMOUNT_REQUESTED']].describe().apply(lambda s: s.apply('{0:.2f}'.format))) 

print("\nTECHNICAL NOTE: Notice the high standard deviation (std) in Income.") 
print("This indicates outliers (people with income way above average).") 
print("Decision: We will use the MEDIAN in data treatment to avoid distortions.") 

In [ ]:
# 3 - Civil Profile Analysis (who are the women?):

# i. Filtering only women:
df_women = df[df['GENDER'] == 'Female'] 

# ii. Counting how many women are in each marital status:
civil_profile = df_women['MARITAL_STATUS'].value_counts() 

print("\nClients' Marital Status Profile (DinDin do Bem):") 
print(civil_profile) 

# iii. Calculating the percentage:
widows_percentage = (civil_profile.get('Widow', 0) / df_women.shape[0]) * 100 
separated_percentage = (civil_profile.get('Separated / Divorced', 0) / df_women.shape[0]) * 100 

print(f"In this database, we have {civil_profile.get('Widow', 0)} widows: ({widows_percentage:.1f}%)") 
print(f"\nAnd we have {civil_profile.get('Separated / Divorced', 0)} separated / divorced: ({separated_percentage:.1f}%)") 
print(f"\nThis proves that this AI will be trained to also serve these vulnerable groups!") 

In [ ]:
# 4.1 - Validating the "Din Din do Bem" fintech hypothesis, where women are better payers than men:

# i. Calculating the 'TARGET' mean (Default Risk) by Gender.
# ii. Multiplying the mean result by 100 to see it as a percentage.
risk_by_gender = df.groupby('GENDER')['TARGET'].mean() * 100 

print("\n ---------- RISK ANALYSIS: MEN vs WOMEN ----------") 
print(f"Default Rate - Men (Male): {risk_by_gender.get('Male', 0):.2f}%") 
print(f"Default Rate - Women (Female): {risk_by_gender.get('Female', 0):.2f}%") 

# 4.2 - Calculating the difference:
difference = risk_by_gender.get('Male', 0) - risk_by_gender.get('Female', 0) 
print(f"\nCONCLUSION: Women have a default rate {difference:.2f}% LOWER. ") 
print("This proves that focusing on women is a safer and more profitable business! ") 

In [ ]:
# 5 - Filtering and Selecting "Alternative Data":
# Here we focus on the fictional fintech's target audience (Women) and select columns that represent BEHAVIOR (phone stability, housing, etc.) instead of just money.

# Creating a new DataFrame (two-dimensional data structure) with only WOMEN:
df_women = df[df['GENDER'] == 'Female'].copy() 

# List of columns that tell the story of "Behavioral Stability":
features = [
    # Social profile (housing, marital status, education):
    'HOUSING_TYPE', 
    'MARITAL_STATUS', 
    'EDUCATION_LEVEL', 

    # Stability (time without changing phone or ID):
    'DAYS_SINCE_PHONE_CHANGE', 
    'DAYS_ID_PUBLISHED', 
    'DAYS_EMPLOYED', # Employment time, even if informal.

    # Financial Capacity:
    'TOTAL_INCOME', 
    'LOAN_AMOUNT_REQUESTED', 

    # Alternative Data (alternative scores from carriers/stores):
    'SOCIAL_MICROCREDIT_SCORE', 
    'BASIC_BILLS_REGISTRY_SCORE', 
    'RETAIL_BEHAVIOR_SCORE' 
]

# Separating X (data for AI to study) and y (answer key: Paid or not?):
X = df_women[features] 
y = df_women['TARGET'] 

print(f"\nFiltered database for DinDin do Bem AI: {X.shape[0]} rows / clients and {X.shape[1]} columns.") 

In [ ]:
# 6 - The Treatment Pipeline (ETL process):

# 6.1 - Automatically identifying which columns are numbers and which are text:
numeric_columns = X.select_dtypes(include=['int64', 'float64']).columns 
text_columns = X.select_dtypes(include=['object']).columns 

# 6.2 - Step-by-step treatment for numbers:
# i. Fill blanks with the MEDIAN;
# ii. Standardize the scale (StandardScaler).
numeric_treatment = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')), 
    ('scaler', StandardScaler()) 
])

# 6.3 - Step-by-step treatment for text:
# i. Fill blanks with 'Unknown';
# ii. Transform into binary columns (One-Hot Encoding):
text_treatment = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')), 
    ('onehot', OneHotEncoder(handle_unknown='ignore')) 
])

# 6.4 - Joining both algorithms into the general processor:
preprocessor = ColumnTransformer(
    transformers = [
        ('num', numeric_treatment, numeric_columns), 
        ('cat', text_treatment, text_columns) 
    ])

print("Applying data treatment (translating to the AI)...") 

X_treated = preprocessor.fit_transform(X) 

print("Data successfully treated!") 

In [ ]:
# 7 - Splitting and Balancing:

# i. Separating data for Training (80%) and Testing (20%):
X_train, X_test, y_train, y_test = train_test_split(X_treated, y, test_size=0.2, random_state=42) 

# ii. Applying SMOTE (only on TRAINING data!)
# We do not touch the Test data, as it must reflect reality.
print("Balancing the data with SMOTE (creating synthetic clones)...") 
smote = SMOTE(random_state=42) 

X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train) 

print(f"Before SMOTE: {y_train.value_counts().to_dict()}") 
print(f"After SMOTE: {y_train_resampled.value_counts().to_dict()}") 
# Now we can see that the numbers for 0 and 1 are equal.

In [ ]:
# 8 - Training the Neural Network (MLP):

print("\nStarting the Neural Network training...") 

neural_network = MLPClassifier(
    hidden_layer_sizes = (100, 50), # Funnel Architecture
    activation = 'relu',            # Standard activation function
    solver = 'adam',                # Optimizer (the "teacher" adjusting weights)
    max_iter = 100,                 # How many times AI will read data (epochs)
    verbose = True,                 # Shows progress
    random_state = 42 
)

# The .fit command is where the magic happens (training):
neural_network.fit(X_train_resampled, y_train_resampled) 
print("Training completed!") 

In [ ]:
# 9 - Final Evaluation:

print("\nPERFORMANCE REPORT:") 

# Asking the AI to predict the test set:
y_pred = neural_network.predict(X_test) 

# Confusion Matrix (Hits and Misses):
print("Confusion Matrix:") 
print(confusion_matrix(y_test, y_pred)) 

# Complete report:
print("\nDetailed Metrics:") 
print(classification_report(y_test, y_pred)) 

print("-" * 30) 
print("READING TIP:") 
print("Look at row '1' (Defaulter) and column 'Recall'.") 
print("The higher this number, the safer your Fintech is against losses.") 

In [ ]:
# 10 - PERFORMANCE EVALUATION (Separate and Explained Charts)

import matplotlib.pyplot as plt 
import seaborn as sns 
from sklearn.metrics import roc_curve, auc, classification_report, confusion_matrix 

# Configuring the visual style
sns.set_style("whitegrid") 

# --- PART 1: METRICS TABLE ---
print("\n" + "="*40) 
print("1. METRICS TABLE (For the Slide)") 
print("="*40) 

# Generating predictions
y_pred = neural_network.predict(X_test) 
y_proba = neural_network.predict_proba(X_test)[:, 1] 

# Creating the beautiful table
report_dict = classification_report(y_test, y_pred, output_dict=True) 
metrics_table = pd.DataFrame(report_dict).transpose() 

# Translating
metrics_table.rename(columns={'precision': 'Precision', 'recall': 'Recall', 'f1-score': 'F1-Score', 'support': 'Qty'}, inplace=True) 
metrics_table.rename(index={'0': 'Good Payer', '1': 'Defaulter', 'accuracy': 'Accuracy'}, inplace=True) 

display(metrics_table.round(2)) 
print("TIP: Focus on the 'Recall' for the 'Defaulter'.") 


# --- PART 2: CONFUSION MATRIX ---
print("\n" + "="*40) 
print("2. CONFUSION MATRIX CHART") 
print("="*40) 

plt.figure(figsize=(8, 6)) # Creates a new and spacious figure
cm = confusion_matrix(y_test, y_pred) 

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, annot_kws={"size": 14}, 
            xticklabels=['Pred: Good', 'Pred: Default'], 
            yticklabels=['Real: Good', 'Real: Default']) 

plt.title('Where did the AI hit and miss?', fontsize=14) 
plt.ylabel('Reality') 
plt.xlabel('AI Prediction') 
plt.show() # Shows the chart alone

print("CHART LEGEND:") 
print("- DARK BLUE Squares: Where the AI got most right.") 
print("- LIGHT Squares: Where the AI missed (False Positives/Negatives).") 


# --- PART 3: ROC CURVE ---
print("\n" + "="*40) 
print("3. ROC CURVE CHART") 
print("="*40) 

fpr, tpr, thresholds = roc_curve(y_test, y_proba) 
roc_auc = auc(fpr, tpr) 

plt.figure(figsize=(8, 6)) # Creates another new figure
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC Curve (AUC = {roc_auc:.2f})') 
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--') # Luck line
plt.xlim([0.0, 1.0]) 
plt.ylim([0.0, 1.05]) 
plt.xlabel('False Positive Rate (Errors)') 
plt.ylabel('True Positive Rate (Hits)') 
plt.title('ROC Curve - Separation Capacity') 
plt.legend(loc="lower right") 
plt.show() 

print("CHART LEGEND:") 
print("- DOTTED Line: Random guessing (50% chance).") 
print("- ORANGE Line: Our AI. The higher and more curved, the better!") 

In [ ]:
# 11 - NEW CLIENT SIMULATION:

# i. Creating the PERSONA (Mrs. Maria)
# Reminder: In the Home Credit database we use, past days are negative. E.g.: -365 = 1 year ago

new_client = pd.DataFrame({
    'HOUSING_TYPE': ['Owned House / Apt'],        # Owns a house (Stability)
    'MARITAL_STATUS': ['Widow'],                  # Widow (vulnerable target audience)
    'EDUCATION_LEVEL': ['High School'], 

    'DAYS_SINCE_PHONE_CHANGE': [-3000],           # Same chip/phone number for 8 years (highly stable)
    'DAYS_ID_PUBLISHED': [-4000],                 # Old ID document (Stable)
    'DAYS_EMPLOYED': [-1500],                     # Working for 4 consecutive years (Constant income)

    'TOTAL_INCOME': [2500.0],                     # Low Income ($ 2,500.00)
    'LOAN_AMOUNT_REQUESTED': [10000.0],           # Requested loan amount: $ 10,000.00

    'SOCIAL_MICROCREDIT_SCORE': [0.60],           # 60% reasonable
    'BASIC_BILLS_REGISTRY_SCORE': [0.65],         # External score (0 to 1) => 65% is reasonable
    'RETAIL_BEHAVIOR_SCORE': [0.70]               # Another Score, 70% is good
})

print("Analyzed client profile: ") 
display(new_client) 

# ii. Applying the same Phase 1 treatment:
# Important Note: Here we use .transform() and not .fit_transform(), as we want to use the rules the AI already learned.

print("\nTranslating new client Mrs. Maria's data to the AI...") 
new_client_treated = preprocessor.transform(new_client) 

# iii. The Prediction (the verdict):

prediction = neural_network.predict(new_client_treated) 
probability = neural_network.predict_proba(new_client_treated) 

# Getting the chance of being a "Good Payer" (Class 0):

chance_to_pay = probability[0][0] * 100 

print('=' * 30) 
print(f"AI VERDICT: {'✅ APPROVED!' if prediction[0] == 0 else '❌ DENIED!'}") 
print(f"AI confidence that the client will pay: {chance_to_pay:.2f}%") 
print('=' * 30) 


if prediction[0] == 0: 
    print("CONCLUSION: DinDin do Bem granted credit to the widow Mrs. Maria!") 
    print("Despite the low income, phone and housing stability weighed in her favor.") 

else: 
    print("CONCLUSION: Credit Denied. The risk was considered high.") 

In [ ]:
# 12. DUEL SIMULATION: MRS. MARIA vs MRS. JOANA:
# Testing two opposite profiles

# Quick function to facilitate testing:

def test_client(client_data, name): 
    # i. Transform data into a format the AI understands:
    client_table = pd.DataFrame(client_data) 
    treated_client = preprocessor.transform(client_table) 

    # ii. AI makes the prediction:
    prediction = neural_network.predict(treated_client) 
    probability = neural_network.predict_proba(treated_client) 
    chance_to_pay = probability[0][0] * 100 

    # iii. Displaying the result on screen:
    print(f"\nCLIENT ANALYSIS: {name}") 
    print(f"Income: $ {client_data['TOTAL_INCOME'][0]:.2f}") 
    print(f"Mobile Line Stability: {abs(client_data['DAYS_SINCE_PHONE_CHANGE'][0])} days with the same number / chip.")
    print(f"Civil Status: {client_data['MARITAL_STATUS'][0]}")

    if prediction[0] == 0: #
        print(f"✅ VERDICT: CREDIT APPROVED! (Confidence: {chance_to_pay:.1f}%)")
    else: #
        print(f"❌ VERDICT: CREDIT DENIED (High Risk). (Chance to pay: only {chance_to_pay:.1f}%)")


# CASE 1: Mrs. Maria (The stable widow - Target Audience)
# Note: High external scores (0.7) and old phone (-3000 days = 8 years)

mrs_maria = {
    'HOUSING_TYPE': ['Owned House / Apt'], 
    'MARITAL_STATUS': ['Widow'], 
    'EDUCATION_LEVEL': ['High School'], 
    'DAYS_SINCE_PHONE_CHANGE': [-3000], 
    'DAYS_ID_PUBLISHED': [-4000], 
    'DAYS_EMPLOYED': [-1500], 
    'TOTAL_INCOME': [2500], 
    'LOAN_AMOUNT_REQUESTED': [10000.0], 
    'SOCIAL_MICROCREDIT_SCORE': [0.60], 
    'BASIC_BILLS_REGISTRY_SCORE': [0.65], 
    'RETAIL_BEHAVIOR_SCORE': [0.70] 
}


# CASE 2: Mrs. Joana (Unstable Profile):
mrs_joana = {
    'HOUSING_TYPE': ['Rented'], 
    'MARITAL_STATUS': ['Single'], 
    'EDUCATION_LEVEL': ['Middle School'],
    'DAYS_SINCE_PHONE_CHANGE': [-10],    # Changed chip last week
    'DAYS_ID_PUBLISHED': [-200],
    'DAYS_EMPLOYED': [-30],              # New job
    'TOTAL_INCOME': [15000.0],           # High Income (catch)
    'LOAN_AMOUNT_REQUESTED': [300000.0], #
    'SOCIAL_MICROCREDIT_SCORE': [0.10],  # Low reputation
    'BASIC_BILLS_REGISTRY_SCORE': [0.20],
    'RETAIL_BEHAVIOR_SCORE': [0.15] 
}

test_client(mrs_maria, "Mrs. Maria (Widow, Low Income)")
test_client(mrs_joana, "Mrs. Joana (Single, High Income)")

# 🏁 Conclusion and Results Analysis
### *Technical and Social Impact of the "DinDin do Bem" Approach*

---

## 📊 Technical Performance of the Model
The development of this project validated the application of **Neural Networks (MLP)** on tabular data for granting credit, demonstrating that it is possible to overcome the barriers of traditional statistical models.

The metrics analysis revealed three pillars of success:
1. **SMOTE Effectiveness:** Applying class balancing was decisive. Without this technique, the model would show a bias toward the majority class ("Good Payers"), ignoring the risk. SMOTE allowed the AI to learn the nuances of the default profile.
2. **Focus on Recall:** In line with the risk strategy, we prioritized the **Recall** metric. In a financial scenario, the ability to detect a "Defaulter" (avoiding losses) is critical. The model demonstrated adequate sensitivity to segregate risks without improperly blocking credit access.
3. **Generalization (ROC Curve):** The area under the curve (AUC) indicated that the classifier has a solid predictive capacity, superior to randomness, validating the choice of behavioral *features*.

## 💡 Business Hypothesis Validation
The practical simulation performed with the personas **"Mrs. Maria"** (Widow, Low Income, High Stability) and **"Mrs. Joana"** (Single, High Income, Low Stability) confirmed the central thesis of *DinDin do Bem*:

> *"Behavior and stability are predictors of trust as strong as formal income."*

The model approved credit for the vulnerable persona (Mrs. Maria) based on her **Alternative Data** (such as phone line stability and basic bills score), proving that technology can act as an engine for financial inclusion, removing the invisibility of women who do not have a pay stub.

## 🚧 Limitations and Future Work
To evolve this MVP (*Minimum Viable Product*) into a market-ready product, we identified the following steps:

* **Explainable AI (XAI):** Implementation of libraries like **SHAP** or **LIME** to make the Neural Network's "black box" transparent, explaining to the client exactly *the reason* for the decision (algorithmic ethics).
* **Open Finance Integration:** Replacing simulated scores with real banking transaction and PIX data, increasing the accuracy of payment capacity.
* **Hyperparameter Optimization:** Performing a *Grid Search* to refine the network architecture (number of hidden layers and neurons), aiming to raise the overall Accuracy without sacrificing Recall.

---
### 🌟 Final Consideration
*DinDin do Bem* demonstrates that Artificial Intelligence, when guided by an ethical purpose and fed by diversified data, is capable of breaking banking paradigms. We transformed **behavioral data into financial dignity**, fulfilling the discipline's objective of applying Machine Learning to solve real and complex problems.